# VeriAudit v2 — inference (Colab / remote GPU)

Runs the models in `configs/models.yaml` with vLLM, in the protocol's order:

1. **Pin model revisions** (the runner refuses unpinned models).
2. **Calibration pilot** (protocol §12): disjoint from the main study; estimates response consistency and heterogeneity, then `run_analysis.py --calibration` re-checks power.
3. **Main study**, only after the dataset is frozen (`assemble_veriaudit_x.py --version 2.0.0 --release`).

Runs are resumable: re-running skips successful generations and retries failed ones (up to `max_attempts`). Requires a GPU runtime and a Hugging Face token with the model licences accepted (Aya Expanse is gated). No paid API is used.


In [ ]:
!git clone -b veriaudit-x-dataset https://github.com/abeeranajam31/veriaudit.git
%cd veriaudit
!pip install -q -r backend/requirements-v2.txt pyyaml vllm
# After the first successful calibration run, pin vllm/torch/transformers in backend/requirements-v2.txt
# to the versions recorded in results/v2/runs/<run_id>/run_manifest.json.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 1. Check pinned revisions

The confirmatory models and the recommended judges were pinned on 2026-11-13 (`configs/models.yaml`, `configs/judges.yaml`). This cell prints the current `main` commit next to each pin. A difference is expected over time and is **not** a reason to change a pin: the study uses the pinned checkpoints.

In [ ]:
import yaml
from huggingface_hub import HfApi
api = HfApi()
for m in yaml.safe_load(open('configs/models.yaml'))['models']:
    if m['backend'] in ('vllm', 'transformers') and m.get('confirmatory'):
        info = api.model_info(m['model_id'])
        print(f"{m['key']:16} {m['model_id']:32} revision: {info.sha}   (currently pinned: {m.get('revision')})")

## 2. Calibration pilot

In [ ]:
# Pre-flight: fails in seconds if a licence, pin or dataset is wrong (run before every GPU session)
!PYTHONPATH=backend python research/inference/pilot_preflight.py

In [ ]:
!PYTHONPATH=backend python research/dataset/build_calibration.py
MODEL_KEY = 'qwen3_8b'  # run each confirmatory model in turn
!PYTHONPATH=backend python -m app.v2.inference.runner configs/experiments/calibration_pilot.yaml --models $MODEL_KEY

### 2b. Judge the pilot (two primary judges, then the tie-breaker on disagreements)

Pin each judge's revision in `configs/judges.yaml` first (as for the models). One judge per session on an 80 GB GPU. Labels go to `results/v2/labels/`; unparseable outputs to `results/v2/labels/failures/` (human queue). Then run the calibration analysis (`docs/v2/calibration_pilot_package.md`).

In [ ]:
JUDGE_KEY = 'gpt_oss_120b'  # then 'mistral_small_3_2'; 'llama4_scout' as tie-breaker
!PYTHONPATH=backend python -m app.v2.judges.run_judge --run results/v2/runs/calibration-pilot-v2 --judge $JUDGE_KEY
!PYTHONPATH=backend python research/analysis/run_analysis.py --run results/v2/runs/calibration-pilot-v2 --calibration --labels results/v2/labels/calibration-pilot-v2__*.jsonl results/v2/labels/human/calibration-pilot-v2__*.jsonl

In [ ]:
# Apply the pre-registered decision rules (A7 sample size; J0 judges). Read pilot_decision.md, then decide.
!PYTHONPATH=backend python research/analysis/pilot_decision.py --analysis results/v2/analysis/calibration-pilot-v2 --run results/v2/runs/calibration-pilot-v2 --labels results/v2/labels/calibration-pilot-v2__*.jsonl results/v2/labels/human/calibration-pilot-v2__*.jsonl
!cat results/v2/analysis/calibration-pilot-v2/pilot_decision.md

## 3. Main study (only after the dataset is frozen)

In [ ]:
MODEL_KEY = 'qwen3_8b'  # one confirmatory model per session
!PYTHONPATH=backend python -m app.v2.inference.runner configs/experiments/main_study.yaml --models $MODEL_KEY

## 4. Check the run before leaving the runtime

Only `status: complete` runs are accepted by the analysis. Copy `results/v2/runs/<run_id>/` (generations.jsonl + run_manifest.json) to Drive or commit it before the runtime ends.

In [ ]:
import json, glob
for p in glob.glob('results/v2/runs/*/run_manifest.json'):
    m = json.load(open(p))
    print(m['run_id'], m['status'], m.get('n_succeeded'), '/', m.get('n_planned'), 'failed:', m.get('n_failed_after_retries'), 'unpinned:', m.get('unpinned_models'))